In [0]:
# initial load for volumn to source table 
from pyspark.sql.functions import current_timestamp, input_file_name, col

volume_path = "/Volumes/dbt_airbnb/source/source_data/"
catalog_name = "dbt_airbnb"
source_schema = "source"
full_schema_path = f"{catalog_name}.{source_schema}"

# 1. Schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {full_schema_path}")

# 2.Volume data
files = dbutils.fs.ls(volume_path)

for file in files:
    if file.name.endswith(".csv"):
        table_name = file.name.replace(".csv", "")
        
        # when read csv, we need to specify schema
        df = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "false") \
            .load(file.path)
        
        df_final = df.withColumn("ingested_at", current_timestamp()) \
                     .withColumn("source_file", col("_metadata.file_path"))
        
        # write to table
        df_final.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(f"{full_schema_path}.{table_name}")

print(f"✅ table {table_name} has been created in Unity Catalog metadata。")

In [0]:
# purpose for incre_load
from pyspark.sql.functions import current_timestamp, col

# set volume path
volume_path = "/Volumes/dbt_airbnb/source/source_data/incre_load/"
catalog_name = "dbt_airbnb"
source_schema = "source"
full_schema_path = f"{catalog_name}.{source_schema}"

files = dbutils.fs.ls(volume_path)

for file in files:
    if file.name.endswith(".csv"):
        
        # union write to table listings ( listing_incre.csv）
        table_name = "listings"
        
        df = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "false") \
            .load(file.path)
        
        df_final = df.withColumn("ingested_at", current_timestamp()) \
                     .withColumn("source_file", col("_metadata.file_path"))
        
        # 👉 append
        df_final.write.format("delta") \
            .mode("append") \
            .saveAsTable(f"{full_schema_path}.{table_name}")

print("✅ data has been appended to table listings")

In [0]:
%sql
select * from dbt_airbnb.source.listings
order by 1

In [0]:
%sql
select count(*) from dbt_airbnb.source.listings
limit 5

In [0]:
%sql
select * from dbt_airbnb.source.listings
where listing_id in (1, 501)

In [0]:
%sql
select * from dbt_airbnb.bronze.stg_bookings

In [0]:
%sql
select * from dbt_airbnb.bronze.stg_listings
limit 5

In [0]:
%sql
SELECT 'stg_bookings' AS table_name, COUNT(*) AS count FROM dbt_airbnb.bronze.stg_bookings
UNION ALL
SELECT 'stg_hosts' AS table_name, COUNT(*) AS count FROM dbt_airbnb.bronze.stg_hosts
UNION ALL
SELECT 'stg_listings' AS table_name, COUNT(*) AS count FROM dbt_airbnb.bronze.stg_listings;

In [0]:
%sql
select * from dbt_airbnb.bronze.stg_hosts
limit 3

In [0]:
%sql
select * from dbt_airbnb.bronze.stg_listings
where listing_id in (1, 501)
order by 1

In [0]:
%sql
select * from dbt_airbnb.silver.silver_hosts
limit 5;

In [0]:
%sql
select * from dbt_airbnb.silver.silver_bookings
limit 5;

In [0]:
%sql
SELECT 
    booking_id, 
    COUNT(*) as occurrences
FROM dbt_airbnb.silver.silver_bookings
GROUP BY 1
HAVING COUNT(*) > 1;

In [0]:
%sql
select * from dbt_airbnb.silver.silver_listings
limit 5;

In [0]:
%sql
select * from dbt_airbnb.gold.obt
where listing_id in (1, 501)
order by 1

In [0]:
%sql
SELECT booking_id, COUNT(*)
FROM dbt_airbnb.gold.obt
GROUP BY booking_id
HAVING COUNT(*) > 1
LIMIT 10;

In [0]:
%sql
select * from dbt_airbnb.gold.obt
limit 5;

In [0]:
%sql
select * from dbt_airbnb.gold.gold_fact_bookings
limit 3


In [0]:
%sql
select * from dbt_airbnb.gold.gold_fact_bookings
where listing_id in (1, 501)
order by 1

In [0]:
%sql
select count(*) 
from dbt_airbnb.snapshots.hosts_snapshot

In [0]:
%sql
select * from dbt_airbnb.snapshots.hosts_snapshot
limit 5

In [0]:
%sql
select * from dbt_airbnb.snapshots.listings_snapshot
where listing_id in (1, 501)
order by 1

In [0]:
%sql
select count(*) 
from dbt_airbnb.snapshots.listings_snapshot

In [0]:
%sql
select * from dbt_airbnb.snapshots.listings_snapshot
limit 5